![image.png](https://i.imgur.com/a3uAqnb.png)
# Dockerizing Foundation Models

This notebook Dockerizes **foundation models** across modalities — CLIP zero-shot classification and YOLO based image segmentation.


> 💡 Foundation models are **broadly pretrained** and **adapted** downstream with little labeled data (linear probes, prompting, fine-tuning).



# 📦 Installing Required Python Libraries

This cell installs packages needed for this lab.

- **Transformers** — CLIP.
- **Torchvision** — Transfer-learning baselines.
- **Pillow / Matplotlib** — Image loading and feature visualization.


In [ ]:
!pip install -q torch torchvision transformers pillow matplotlib numpy scipy scikit-learn


# 📥 Importing Essential Python Libraries

Imports for CLIP experiments and sklearn linear probes.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


### 🛠️ CLIP zero-shot classification

 **CLIP** aligns image and text in a shared embedding space. At inference, cosine similarity between an image and text prompts yields **zero-shot** classification — no task-specific fine-tuning required.

In [ ]:
try:
    from transformers import CLIPProcessor, CLIPModel
    from PIL import Image
    import requests

    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    labels = ["a photo of Zinadine Zidane", "a photo of Cristiano Ronaldo", "a photo of Harry Kane"]
    try:
        url = "https://ultralytics.com/images/zidane.jpg"
        image = Image.open(requests.get(url, stream=True, timeout=30).raw).convert("RGB")
    except Exception:
        image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))

    # Display the image
    plt.imshow(image)
    plt.title("Image for CLIP classification")
    plt.axis("off")
    plt.show()

    inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=-1)[0]
    pred = probs.argmax().item()
    print("Predicted:", labels[pred])
    for label, pval in zip(labels, probs.tolist()):
        print(f"  {label}: {pval:.3f}")
except Exception as exc:
    print(f"CLIP demo skipped ({exc}).")
    labels = ["cat", "dog", "car"]
    logits = torch.tensor([2.0, 0.5, -1.0])
    probs = logits.softmax(0)
    print("Predicted (toy):", labels[probs.argmax()])

# Setup

pip install `ultralytics` and [dependencies](https://github.com/ultralytics/ultralytics/blob/main/pyproject.toml) and check software and hardware.

In [ ]:
!uv pip install ultralytics
import ultralytics
ultralytics.checks()

# Predict

YOLO26 may be used directly in the Command Line Interface (CLI) with a `yolo` command for a variety of tasks and modes and accepts additional arguments, i.e. `imgsz=640`. See a full list of available `yolo` [arguments](https://docs.ultralytics.com/usage/cfg/) and other details in the [YOLO26 Predict Docs](https://docs.ultralytics.com/modes/predict/).

In [ ]:
# Run inference on an image with YOLO26n
!yolo predict model=yolo26n.pt source='https://ultralytics.com/images/zidane.jpg'

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<img align="left" src="https://user-images.githubusercontent.com/26833433/212889447-69e5bdf1-5800-4e29-835e-2ed2336dede2.jpg" width="600">

# Dockerization Task

**Task:** Choose one of the foundational models demonstrated in this notebook (either CLIP for zero-shot classification or YOLO for object detection) and Dockerize it.

**Instructions:**
1.  **Choose a Model:** Select either the CLIP model or the YOLO model.
2.  **Create a `Dockerfile`:** Write a `Dockerfile` that:
    *   Sets up the necessary Python environment (e.g., `FROM python:3.9-slim-buster`).
    *   Installs all required dependencies (e.g., `pip install torch torchvision transformers` for CLIP, or `pip install ultralytics` for YOLO).
    *   Copies your inference script into the Docker image.
    *   Defines the entry point to run your inference script.
3.  **Develop an Inference Script:** Create a Python script that loads the chosen model, takes an input (e.g., an image URL or local path), performs inference, and prints the result.
4.  **Build and Run the Docker Image:**
    *   Build your Docker image locally using `docker build -t <your-image-name> .`
    *   Run your Docker container using `docker run <your-image-name>` to test your Dockerized model.